# Kilonova transformer — entrenamiento local (OpenUniverse, PyTorch Lightning, GPU)

Clasificación **binaria `{KN, other}`** sobre el dataset OpenUniverse de ventanas tempranas.
Cuatro fuentes: `kilonova_windows_{deep,wide}.hdf5` (la clase KN) y `early_windows_{deep,wide}.parquet`
(los contaminantes: SN II/Ia/Ib/Ic/Iax, TDE, SLSN-I, PISN, todos agrupados como `other`).
**deep y wide se combinan** en un solo modelo sobre un vocabulario de 6 bandas
(R062, Z087, Y106, J129, H158, F184).

**Split 90/5/5 sin leakage:** las KN se separan por `simulation_id` (el modelo físico de eyecta;
un modelo vive en un solo split, global deep+wide), los contaminantes por `object_id` estratificado
por clase original. ⚠️ El parquet no guarda el template SED de SNANA (`snana_id` es un id de batch,
no el modelo), así que el leakage de template entre core-collapse no se puede blindar desde estos datos.

Igual que antes: scheduler **cosine con warmup**, **EarlyStopping**, **validación balanceada por régimen**
({1,2,3 épocas} × {con z, sin z}). La selección de modelo usa **`val_acc_z`**; se reporta **`val_acc_noz`**
aparte. La normalización de magnitudes se ajusta **solo en el split de train**.

GPU (RTX 4060 Laptop, 8 GB): las secuencias son minúsculas (≤22 tokens), la memoria no es el cuello
de botella → `batch_size=1024`, `precision='bf16-mixed'`, `num_workers=8`.

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from openuniverse_data import build_dataloaders, GROUP_ORDER

DATA_DIR = '/home/nicolas/nico/git/Kilonova/data/openuniverse'

data = build_dataloaders(
    deep_hdf5=f'{DATA_DIR}/kilonova_windows_deep.hdf5',
    wide_hdf5=f'{DATA_DIR}/kilonova_windows_wide.hdf5',
    deep_parquet=f'{DATA_DIR}/early_windows_deep.parquet',
    wide_parquet=f'{DATA_DIR}/early_windows_wide.parquet',
    batch_size=512,
    num_workers=8,
    cache_path=f'{DATA_DIR}/openuniverse_tokens.npz',  # primera corrida ~6 min; luego instantáneo
)
train_loader = data['train_loader']
regime_loaders = data['validation_regime_loaders']
validation_loaders = [r['loader'] for r in regime_loaders]
regime_names = [r['name'] for r in regime_loaders]
print('split:', data['split_sizes'])
print('balance:', data['class_balance'])
print('regimenes de validacion:', regime_names)

In [ ]:
import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from train_lightning import LitKilonova

L.seed_everything(0)
MAX_EPOCHS = 150

# class weights desde el conteo del split (sin recorrer el loader). 'other' es la mayoria (~1.6M)
# y KN la minoria (~324k); 'sqrt' suaviza el peso. Usa mode='inverse' o 'none' para comparar.
def class_weights(counts, mode='sqrt'):
    counts = torch.tensor(counts, dtype=torch.float)
    inverse = counts.sum() / (len(counts) * counts)
    if mode == 'sqrt':
        inverse = inverse.sqrt()
    elif mode == 'none':
        inverse = torch.ones_like(counts)
    return inverse

train_counts = [data['class_balance']['train'][name] for name in GROUP_ORDER]
weights = class_weights(train_counts, mode='sqrt')
print('class weights', GROUP_ORDER, [round(w, 3) for w in weights.tolist()])

# Modelo mas grande (la GPU 8GB tiene holgura de sobra con secuencias <=22 tokens).
lit_model = LitKilonova(
    class_weights=weights, learning_rate=1e-3, weight_decay=1e-4,
    d_model=192, num_heads=6, num_layers=6, d_feedforward=768, dropout=0.1,
    max_epochs=MAX_EPOCHS, warmup_epochs=5, val_regime_names=regime_names,
)
print('parametros:', sum(p.numel() for p in lit_model.model.parameters()))

# Seleccionamos por val_acc_noz (sin z): KN y contaminantes estan casi disjuntos en redshift,
# asi que val_acc_z=1.0 es artefacto. Guardamos top-5 para promediar pesos (model soup) y no
# depender de una sola epoca (que puede ser un outlier de validacion).
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='kn_transformer-{epoch:02d}-{val_acc_noz:.4f}',
    monitor='val_acc_noz', mode='max', save_top_k=5,
)
early_stopping = EarlyStopping(monitor='val_acc_noz', mode='max', patience=25)
lr_monitor = LearningRateMonitor(logging_interval='epoch')

In [ ]:
trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    devices='auto',
    precision='bf16-mixed',  # Ada (RTX 4060): mas rapido y libera memoria
    callbacks=[checkpoint_callback, early_stopping, lr_monitor],
    log_every_n_steps=10,
)
trainer.fit(lit_model, train_loader, validation_loaders)
print('best val_acc_noz:', float(checkpoint_callback.best_model_score),
      '->', checkpoint_callback.best_model_path)

## Desglose por régimen y por clase del mejor checkpoint

Cada régimen es determinista. Aquí ves el accuracy macro por régimen (1/2/3 épocas, con/sin z)
y la matriz de confusión, para ver cómo se separan `Ia` y `CCSN` en cada régimen.

In [ ]:
from train_lightning import MODEL_INPUT_KEYS
from torchmetrics.classification import MulticlassAccuracy, MulticlassConfusionMatrix

n_classes = len(GROUP_ORDER)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
best = LitKilonova.load_from_checkpoint(checkpoint_callback.best_model_path,
                                        class_weights=weights).to(device).eval()
confusion_3ep_z = MulticlassConfusionMatrix(num_classes=n_classes).to(device)

print(f'{"regimen":>10}  {"acc_macro":>9}')
for regime in regime_loaders:
    acc = MulticlassAccuracy(num_classes=n_classes).to(device)
    with torch.no_grad():
        for batch in regime['loader']:
            b = {k: v.to(device) for k, v in batch.items() if k in MODEL_INPUT_KEYS}
            logits = best(b); y = b['label']
            acc(logits, y)
            if regime['name'] == '3ep_z':
                confusion_3ep_z(logits, y)
    print(f'{regime["name"]:>10}  {acc.compute().item():>9.4f}')

print('\nmatriz de confusion 3ep_z (filas=verdad, cols=pred)', GROUP_ORDER)
print(confusion_3ep_z.compute().long().cpu().numpy())